In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---

## Level 1 — MPG

Use `sns.load_dataset('mpg').dropna()`. Group by `origin`.

Named-agg summary (your choice of syntax): `n`, `avg_mpg` (mean), `avg_horsepower` (mean), `weight_spread` (max minus min of `weight`).

- Rank origins from most to least fuel-efficient using `np.argsort`.
- Is higher average weight associated with lower mpg across origins? Print the correlation.
- Build `{origin: avg_mpg}` using a dict comprehension.

In [7]:
mpg = sns.load_dataset('mpg').dropna()

# Your code here
g = mpg.groupby('origin').agg(
    n = ('mpg','count'),
    avg_mpg = ('mpg','mean'),
    avg_horsepower = ('horsepower','mean'),
    weight_spread = pd.NamedAgg(column='weight',aggfunc=(lambda x: x.max() - x.min())),
    avg_weight = ('weight','mean')
)

print(g.index[np.argsort(-g['avg_mpg'])])

cor = np.corrcoef(g['avg_weight'],g['avg_mpg'])[0,1]
print('cor is ',cor)
print('higher avg weight associates with lower mpg')

{o:am for o, am in zip(g.index, g['avg_mpg'])}

Index(['japan', 'europe', 'usa'], dtype='object', name='origin')
cor is  -0.9956269159912966
higher avg weight associates with lower mpg


{'europe': 27.602941176470587,
 'japan': 30.450632911392404,
 'usa': 20.0334693877551}

---

## Level 2 — Library loans

`loans` records individual book loans across three branches and four genres. Before grouping, add `overdue = (days_borrowed > 14).astype(int)`.

Group by `branch` and `genre`. Named agg: `total_loans` (count), `avg_days` (mean of `days_borrowed`), `return_rate` (mean of `is_returned`), `overdue_rate` (mean of `overdue`). Then add `efficiency = return_rate / avg_days` as a post-agg column.

- Which (`branch`, `genre`) pair has the highest efficiency? The highest overdue rate?
- What are the 10th and 90th percentiles of `avg_days` across all groups?
- Is there a relationship between `overdue_rate` and `return_rate`? Print the correlation.

In [18]:
np.random.seed(9)
k = 240
loans = pd.DataFrame({
    'branch':       np.random.choice(['Central', 'North', 'East'], k),
    'genre':        np.random.choice(['Fiction', 'Non-Fiction', 'Science', 'History'], k),
    'days_borrowed': np.random.randint(1, 30, k),
    'is_returned':  np.random.choice([0, 1, 1, 1], k),
})

# Your code here

loans['overdue'] = (loans['days_borrowed']>14).astype(int)

g = loans.groupby(['branch','genre']).agg(
    total_loans = ('is_returned','count'),
    avg_days = ('days_borrowed','mean'),
    return_rate = ('is_returned','mean'),
    overdue_rate = ('overdue','mean')
)

g['efficiency'] = g['return_rate']/g['avg_days']

print(g['efficiency'].idxmax(),'has the highest efficiency')
print(g['overdue_rate'].idxmax(),'has the highest overdue rate')
print(np.percentile(g['avg_days'],q = [10,90]))
cor = np.corrcoef(g['overdue_rate'], g['return_rate'])[0,1]
print(f'correlation is {cor:.2f}')
print('not very well correlated')

('Central', 'Fiction') has the highest efficiency
('East', 'Fiction') has the highest overdue rate
[11.63194444 17.43171429]
correlation is 0.03
not very well correlated


---

## Level 3 — Titanic

Use `sns.load_dataset('titanic')`. Group by `pclass` and `embark_town` (`observed=True`).

Named-agg summary: `survival_rate` (mean of `survived`), `avg_fare` (mean), `avg_age` (mean of `age`), `n` (count). No hints below.

- Which (`pclass`, `embark_town`) combination had the highest survival rate? The lowest?
- Unstack by `embark_town`. Across embarkation towns, which passenger class had the most variable survival rate?
- Does average fare correlate with survival rate across groups? Print the value.
- Rank all groups by `avg_fare` and by `survival_rate`. Do the two rankings agree? Quantify it.

In [31]:
titanic = sns.load_dataset('titanic')

# Your code here

g = titanic.groupby(['pclass','embark_town'], observed=True).agg(
    survival_rate = ('survived','mean'),
    avg_fare = ('fare','mean'),
    avg_age = ('age','mean'),
    n = ('fare','count')

)

print(g['survival_rate'].idxmax(),'has the highest survival rate')
print(g['survival_rate'].idxmin(), 'has the lowest survival rate')

sr =g['survival_rate'].unstack()
print(sr.std(axis=1).idxmax(),'has the most variable survival rate')

cor = np.corrcoef(g['avg_fare'],g['survival_rate'])[0,1]
print('correlation is ',cor)
print('positively correlated')


ra = np.argsort(g['avg_fare'])
rs = np.argsort(g['survival_rate'])

rcor = np.corrcoef(ra, rs)[0,1]

print('correlation between ranks is', rcor)
print('positively correlated')


(np.int64(1), 'Cherbourg') has the highest survival rate
(np.int64(3), 'Southampton') has the lowest survival rate
3 has the most variable survival rate
correlation is  0.5421616772732818
positively correlated
correlation between ranks is 0.6666666666666667
positively correlated
